# Case 1: Telecom Customer Churn

**IITGN Cohort 2 · Week 5, Day 2 · Practical (teaching copy)**

## The Business Question

A phone and internet company is losing customers. Marketing can send a retention offer that costs **Rs 500** per customer. A customer who leaves costs the company about **Rs 6,000** in lost revenue.

**Who should receive the offer?**

## The Data

- Public IBM sample dataset of 7,043 real customers.
- Each row is one customer. The column `Churn` says whether the customer left.
- The file has 21 columns. We use only 8 of them, all with plain meanings.

## The Eight Steps

1. Load and look
2. Find broken values
3. Fix them, on a few rows first, then on all rows
4. Prepare the columns
5. Split into train and test
6. Train the model and predict
7. Score the model
8. Move the cut-off and decide

## Step 0: Import the Libraries

We import everything we need in one place, at the top.

In [1]:
# pandas: to read and work with tables
import pandas as pd

# train_test_split: to split rows into a training part and a test part
from sklearn.model_selection import train_test_split

# LogisticRegression: the model we will train
from sklearn.linear_model import LogisticRegression

# Functions to score the model
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

## Step 1: Load and Look at the Data

Before changing anything, we read the file and look at it.

### 1.1 Read the File

In [2]:
# The file is stored on GitHub. pandas can read it straight from the link.
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

### 1.2 How Many Rows and Columns?

In [3]:
# shape gives (number of rows, number of columns)
df.shape

(7043, 21)

**Result.** 7,043 customers and 21 columns.

### 1.3 Look at the First Five Rows

In [4]:
# head() shows the first five rows
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 1.4 Keep Only the Columns We Need

In [5]:
# The 8 input columns we will use
feature_cols = [
    "tenure",           # months the customer has been with the company
    "MonthlyCharges",   # amount billed every month
    "TotalCharges",     # total amount billed so far
    "SeniorCitizen",    # 1 if the customer is a senior citizen, else 0
    "Contract",         # Month-to-month, One year or Two year
    "InternetService",  # DSL, Fiber optic or No
    "PaymentMethod",    # how the customer pays
    "TechSupport",      # Yes, No or No internet service
]

# The column we want to predict
target_col = "Churn"

# Keep the 8 inputs plus the target
df = df[feature_cols + [target_col]]

df.head()

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport,Churn
0,1,29.85,29.85,0,Month-to-month,DSL,Electronic check,No,No
1,34,56.95,1889.5,0,One year,DSL,Mailed check,No,No
2,2,53.85,108.15,0,Month-to-month,DSL,Mailed check,No,Yes
3,45,42.30,1840.75,0,One year,DSL,Bank transfer (automatic),Yes,No
4,2,70.70,151.65,0,Month-to-month,Fiber optic,Electronic check,No,Yes


### 1.5 How Many Customers Left?

In [6]:
# Count of each answer in the Churn column
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [7]:
# The same counts as a share of all customers
df["Churn"].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

**Result.** 1,869 of 7,043 customers left, which is 26.5 percent. The other 73.5 percent stayed. Remember 73.5: it comes back in Step 7.

### 1.6 Does the Contract Type Matter?

In [8]:
# For each contract type, the share of customers who left
df.groupby("Contract")["Churn"].value_counts(normalize=True)

Contract        Churn
Month-to-month  No       0.572903
                Yes      0.427097
One year        No       0.887305
                Yes      0.112695
Two year        No       0.971681
                Yes      0.028319
Name: proportion, dtype: float64

**Result.** About 43 percent of month-to-month customers left, against about 3 percent of two-year customers. Contract type will be one of the strongest inputs.

## Step 2: Find Broken or Missing Values

We check two things: blank cells, and columns stored in the wrong type.

### 2.1 Count Blank Cells in Each Column

In [9]:
# isna() marks each blank cell as True. sum() counts them per column.
df.isna().sum()

tenure             0
MonthlyCharges     0
TotalCharges       0
SeniorCitizen      0
Contract           0
InternetService    0
PaymentMethod      0
TechSupport        0
Churn              0
dtype: int64

**Result.** Every count is zero. It looks clean, but we are not done.

### 2.2 Check the Type of Each Column

In [10]:
# dtypes shows how pandas stored each column
# int64 and float64 are numbers. object (or str) means text.
df.dtypes

tenure               int64
MonthlyCharges     float64
TotalCharges           str
SeniorCitizen        int64
Contract               str
InternetService        str
PaymentMethod          str
TechSupport            str
Churn                  str
dtype: object

**Result.** `TotalCharges` is stored as text (`object`, or `str` in newer pandas). A money column should be a number. Something inside it is not a number.

### 2.3 Find the Values That Are Not Numbers

In [11]:
# to_numeric converts text to numbers.
# errors="coerce" turns anything that is not a number into a blank (NaN).
total_as_number = pd.to_numeric(df["TotalCharges"], errors="coerce")

# How many values could not be converted?
total_as_number.isna().sum()

np.int64(11)

### 2.4 Look at Those Rows

In [12]:
# Show the rows where the conversion failed
df[total_as_number.isna()]

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport,Churn
488,0,52.55,,0,Two year,DSL,Bank transfer (automatic),Yes,No
753,0,20.25,,0,Two year,No,Mailed check,No internet service,No
936,0,80.85,,0,Two year,DSL,Mailed check,No,No
1082,0,25.75,,0,Two year,No,Mailed check,No internet service,No
1340,0,56.05,,0,Two year,DSL,Credit card (automatic),Yes,No
3331,0,19.85,,0,Two year,No,Mailed check,No internet service,No
3826,0,25.35,,0,Two year,No,Mailed check,No internet service,No
4380,0,20.00,,0,Two year,No,Mailed check,No internet service,No
5218,0,19.70,,0,One year,No,Mailed check,No internet service,No
6670,0,73.35,,0,Two year,DSL,Mailed check,Yes,No


**Result.** 11 customers have a blank space instead of a number. All of them have `tenure` equal to 0: they joined this month and have not been billed yet. So their total charges should be 0.

## Step 3: Fix the Broken Values

We first fix a small sample of 10 rows, so we can see every change. Then we run the same lines on all rows.

### 3.1 Make a Small Sample of 10 Rows

In [13]:
# 7 normal rows from the top, plus 3 of the new customers with tenure 0
small = pd.concat([df.head(7), df[df["tenure"] == 0].head(3)])

small

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport,Churn
0,1,29.85,29.85,0,Month-to-month,DSL,Electronic check,No,No
1,34,56.95,1889.5,0,One year,DSL,Mailed check,No,No
2,2,53.85,108.15,0,Month-to-month,DSL,Mailed check,No,Yes
3,45,42.30,1840.75,0,One year,DSL,Bank transfer (automatic),Yes,No
4,2,70.70,151.65,0,Month-to-month,Fiber optic,Electronic check,No,Yes
5,8,99.65,820.5,0,Month-to-month,Fiber optic,Electronic check,No,Yes
6,22,89.10,1949.4,0,Month-to-month,Fiber optic,Credit card (automatic),No,No
488,0,52.55,,0,Two year,DSL,Bank transfer (automatic),Yes,No
753,0,20.25,,0,Two year,No,Mailed check,No internet service,No
936,0,80.85,,0,Two year,DSL,Mailed check,No,No


### 3.2 Convert TotalCharges to Numbers in the Sample

In [14]:
# The blank spaces become NaN, so we can see them
small["TotalCharges"] = pd.to_numeric(small["TotalCharges"], errors="coerce")

small

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport,Churn
0,1,29.85,29.85,0,Month-to-month,DSL,Electronic check,No,No
1,34,56.95,1889.50,0,One year,DSL,Mailed check,No,No
2,2,53.85,108.15,0,Month-to-month,DSL,Mailed check,No,Yes
3,45,42.30,1840.75,0,One year,DSL,Bank transfer (automatic),Yes,No
4,2,70.70,151.65,0,Month-to-month,Fiber optic,Electronic check,No,Yes
5,8,99.65,820.50,0,Month-to-month,Fiber optic,Electronic check,No,Yes
6,22,89.10,1949.40,0,Month-to-month,Fiber optic,Credit card (automatic),No,No
488,0,52.55,NaN,0,Two year,DSL,Bank transfer (automatic),Yes,No
753,0,20.25,NaN,0,Two year,No,Mailed check,No internet service,No
936,0,80.85,NaN,0,Two year,DSL,Mailed check,No,No


### 3.3 Fill the Blanks with 0 in the Sample

In [15]:
# A new customer has paid nothing yet, so 0 is the correct value
small["TotalCharges"] = small["TotalCharges"].fillna(0)

small

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport,Churn
0,1,29.85,29.85,0,Month-to-month,DSL,Electronic check,No,No
1,34,56.95,1889.50,0,One year,DSL,Mailed check,No,No
2,2,53.85,108.15,0,Month-to-month,DSL,Mailed check,No,Yes
3,45,42.30,1840.75,0,One year,DSL,Bank transfer (automatic),Yes,No
4,2,70.70,151.65,0,Month-to-month,Fiber optic,Electronic check,No,Yes
5,8,99.65,820.50,0,Month-to-month,Fiber optic,Electronic check,No,Yes
6,22,89.10,1949.40,0,Month-to-month,Fiber optic,Credit card (automatic),No,No
488,0,52.55,0.00,0,Two year,DSL,Bank transfer (automatic),Yes,No
753,0,20.25,0.00,0,Two year,No,Mailed check,No internet service,No
936,0,80.85,0.00,0,Two year,DSL,Mailed check,No,No


**Result.** The last three rows now show 0, and the column is a number column. The fix works on the sample.

### 3.4 Apply the Same Two Lines to All Rows

In [16]:
# Same two lines as above, now on the full table
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

### 3.5 Check the Fix

In [17]:
# No blanks should remain
df.isna().sum()

tenure             0
MonthlyCharges     0
TotalCharges       0
SeniorCitizen      0
Contract           0
InternetService    0
PaymentMethod      0
TechSupport        0
Churn              0
dtype: int64

In [18]:
# TotalCharges should now be float64, a number
df.dtypes

tenure               int64
MonthlyCharges     float64
TotalCharges       float64
SeniorCitizen        int64
Contract               str
InternetService        str
PaymentMethod          str
TechSupport            str
Churn                  str
dtype: object

## Step 4: Prepare the Columns

The model reads only numbers. We turn the target into 1 and 0, and turn the text columns into 0/1 columns.

### 4.1 Turn Churn into 1 and 0, in the Sample First

In [19]:
# Yes becomes 1, No becomes 0
small["Churn"] = small["Churn"].map({"Yes": 1, "No": 0})

small[["Churn"]]

,Churn
0,0
1,0
2,1
3,0
4,1
5,1
6,0
488,0
753,0
936,0


### 4.2 Turn Churn into 1 and 0 for All Rows

In [20]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Check: we should see only 0 and 1
df["Churn"].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

### 4.3 Turn Text Columns into 0/1 Columns, in the Sample First

In [21]:
# Look at 5 rows of the sample before the change
small[feature_cols].head(5)

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract,InternetService,PaymentMethod,TechSupport
0,1,29.85,29.85,0,Month-to-month,DSL,Electronic check,No
1,34,56.95,1889.50,0,One year,DSL,Mailed check,No
2,2,53.85,108.15,0,Month-to-month,DSL,Mailed check,No
3,45,42.30,1840.75,0,One year,DSL,Bank transfer (automatic),Yes
4,2,70.70,151.65,0,Month-to-month,Fiber optic,Electronic check,No


In [22]:
# get_dummies makes one 0/1 column for each category of each text column.
# drop_first=True drops one column per text column, because it can be worked out from the others.
# dtype=int shows 0 and 1 instead of False and True.
pd.get_dummies(small[feature_cols].head(5), drop_first=True, dtype=int)

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract_One year,InternetService_Fiber optic,PaymentMethod_Electronic check,PaymentMethod_Mailed check,TechSupport_Yes
0,1,29.85,29.85,0,0,0,1,0,0
1,34,56.95,1889.50,0,1,0,0,1,0
2,2,53.85,108.15,0,0,0,0,1,0
3,45,42.30,1840.75,0,1,0,0,0,1
4,2,70.70,151.65,0,0,1,1,0,0


**Result.** `Contract` has become `Contract_One year` and `Contract_Two year`. A month-to-month customer has 0 in both. The number columns stay as they are.

### 4.4 Build the Inputs (X) and the Target (y) for All Rows

In [23]:
# X: the inputs, with text columns turned into 0/1 columns
X = pd.get_dummies(df[feature_cols], drop_first=True, dtype=int)

# y: the answer we want to predict
y = df["Churn"]

X.shape

(7043, 13)

In [24]:
# The final list of input columns the model will see
X.columns

Index(['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen',
       'Contract_One year', 'Contract_Two year', 'InternetService_Fiber optic',
       'InternetService_No', 'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'TechSupport_No internet service', 'TechSupport_Yes'],
      dtype='str')

In [25]:
X.head()

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract_One year,Contract_Two year,InternetService_Fiber optic,InternetService_No,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TechSupport_No internet service,TechSupport_Yes
0,1,29.85,29.85,0,0,0,0,0,0,1,0,0,0
1,34,56.95,1889.50,0,1,0,0,0,0,0,1,0,0
2,2,53.85,108.15,0,0,0,0,0,0,0,1,0,0
3,45,42.30,1840.75,0,1,0,0,0,0,0,0,0,1
4,2,70.70,151.65,0,0,0,1,0,0,1,0,0,0


## Step 5: Split into Train and Test

The model learns from 75 percent of the rows. We keep 25 percent aside as the final exam.

### 5.1 Split the Rows

In [26]:
# test_size=0.25: one quarter of the rows go to the test set
# random_state=42: everyone in the room gets the same split
# stratify=y: the share of leavers is the same in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

### 5.2 Check the Sizes

In [27]:
print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

Training rows: 5282
Test rows: 1761


### 5.3 Check the Share of Leavers in Each Part

In [28]:
print("Share of leavers in training rows:", round(y_train.mean(), 3))
print("Share of leavers in test rows:", round(y_test.mean(), 3))

Share of leavers in training rows: 0.265
Share of leavers in test rows: 0.265


**Result.** 5,282 rows to train and 1,761 to test. Both parts have about 26.5 percent leavers, thanks to `stratify`.

## Step 6: Train the Model and Predict

### 6.1 Create and Train the Model

In [29]:
# max_iter=2000 gives the model enough steps to finish learning
model = LogisticRegression(max_iter=2000)

# fit() learns one weight for each input column, from the training rows only
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

### 6.2 Look at the Weights

In [30]:
# One weight per input column. Sorted from most negative to most positive.
weights = pd.Series(model.coef_[0], index=X.columns)

weights.sort_values()

Contract_Two year                       -1.396914
Contract_One year                       -0.758918
InternetService_No                      -0.455200
TechSupport_No internet service         -0.455200
TechSupport_Yes                         -0.425860
tenure                                  -0.053933
PaymentMethod_Credit card (automatic)   -0.023885
TotalCharges                             0.000283
MonthlyCharges                           0.002657
PaymentMethod_Mailed check               0.027744
SeniorCitizen                            0.293164
PaymentMethod_Electronic check           0.444331
InternetService_Fiber optic              0.746509
dtype: float64

**Result.** A negative weight pulls the churn score down. A positive weight pushes it up. The contract columns have strong negative weights: customers on longer contracts leave less.

### 6.3 Get a Score for Each Test Customer

In [31]:
# predict_proba gives two columns: chance of 0 (stays) and chance of 1 (leaves).
# [:, 1] keeps only the chance of leaving.
scores = model.predict_proba(X_test)[:, 1]

# The first 10 scores
scores[:10]

array([0.13974747, 0.05489114, 0.07220066, 0.71693275, 0.01476117,
       0.70415696, 0.19222959, 0.02022714, 0.61538466, 0.65638638])

### 6.4 Turn Scores into Yes or No with the Default Cut-off

In [32]:
# predict() uses a cut-off of 0.5 without telling you
y_pred_05 = model.predict(X_test)

# The first 10 predictions
y_pred_05[:10]

array([0, 0, 0, 1, 0, 1, 0, 0, 1, 1])

In [33]:
# Check: drawing the line at 0.5 ourselves gives the same answers
y_pred_check = (scores >= 0.5).astype(int)

# True means every single prediction matches
(y_pred_check == y_pred_05).all()

np.True_

## Step 7: Score the Model

First the baseline, then accuracy, then the four boxes, then precision and recall.

### 7.1 The Baseline: What If We Say Nobody Leaves?

In [34]:
# Share of each answer in the test rows
y_test.value_counts(normalize=True)

Churn
0    0.73481
1    0.26519
Name: proportion, dtype: float64

**Result.** A rule that says "nobody leaves" would be right for 73.5 percent of test customers. Any model must beat this.

### 7.2 Accuracy of Our Model

In [35]:
accuracy_05 = accuracy_score(y_test, y_pred_05)

print("Accuracy at cut-off 0.5:", round(accuracy_05, 3))

Accuracy at cut-off 0.5: 0.798


**Result.** About 79.8 percent. Better than 73.5 percent, but only by about 6 points. Accuracy alone does not tell us how many leavers we caught.

### 7.3 The Four Boxes

In [36]:
# confusion_matrix(real answers, predicted answers)
# sklearn prints the No row first:
# [[correct no, false alarm],
#  [missed,     caught     ]]
cm_05 = confusion_matrix(y_test, y_pred_05)

cm_05

array([[1155,  139],
       [ 217,  250]])

In [37]:
# ravel() lays the four numbers out in one line, in this order: TN, FP, FN, TP
tn_05, fp_05, fn_05, tp_05 = cm_05.ravel()

print("Caught (TP):", tp_05)
print("False alarms (FP):", fp_05)
print("Missed (FN):", fn_05)
print("Correct no (TN):", tn_05)

Caught (TP): 250
False alarms (FP): 139
Missed (FN): 217
Correct no (TN): 1155


**Result.** Of the 467 test customers who really left, the model caught 250 and missed 217. It also raised 139 false alarms.

### 7.4 Precision, by Hand

In [38]:
# Of those we flagged, how many really left?
precision_05 = tp_05 / (tp_05 + fp_05)

print("Precision at cut-off 0.5:", round(precision_05, 3))

Precision at cut-off 0.5: 0.643


In [39]:
# Check with sklearn: the number should be the same
precision_score(y_test, y_pred_05)

0.6426735218508998

### 7.5 Recall, by Hand

In [40]:
# Of those who really left, how many did we catch?
recall_05 = tp_05 / (tp_05 + fn_05)

print("Recall at cut-off 0.5:", round(recall_05, 3))

Recall at cut-off 0.5: 0.535


In [41]:
# Check with sklearn: the number should be the same
recall_score(y_test, y_pred_05)

0.5353319057815846

**Result.** Precision is about 64 percent: of every 100 customers we flag, 64 really leave. Recall is about 54 percent: we miss nearly half of the leavers.

### 7.6 The Say-No Rule, Scored the Same Way

In [42]:
# A prediction of 0 (stays) for every test customer
y_pred_say_no = [0] * len(y_test)

print("Accuracy of the say-no rule:", round(accuracy_score(y_test, y_pred_say_no), 3))
print("Recall of the say-no rule:", recall_score(y_test, y_pred_say_no))

Accuracy of the say-no rule: 0.735
Recall of the say-no rule: 0.0


**Result.** 73.5 percent accurate and 0 percent recall. It catches nobody. This is the accuracy trap.

## Step 8: Move the Cut-off and Decide

We price each mistake, then compare three cut-offs: 0.5, 0.3 and 0.2.

### 8.1 The Cost of Each Mistake

In [43]:
# A false alarm: we send an offer to a customer who would have stayed
cost_false_alarm = 500

# A miss: a customer leaves and we did nothing
cost_miss = 6000

# Assumption: an offer to a real leaver keeps them, so it pays for itself.

### 8.2 The Bill at Cut-off 0.5

In [44]:
bill_05 = fp_05 * cost_false_alarm + fn_05 * cost_miss

print("False alarms:", fp_05, "  Misses:", fn_05)
print("Bill at cut-off 0.5: Rs", bill_05)

False alarms: 139   Misses: 217
Bill at cut-off 0.5: Rs 1371500


### 8.3 The Bill If We Do Nothing

In [45]:
# Doing nothing means every leaver is a miss and there are no false alarms
fn_nothing = y_test.sum()
bill_nothing = fn_nothing * cost_miss

print("Leavers in the test rows:", fn_nothing)
print("Bill if we do nothing: Rs", bill_nothing)

Leavers in the test rows: 467
Bill if we do nothing: Rs 2802000


**Result.** At 0.5 the bill is Rs 13,71,500. Doing nothing costs Rs 28,02,000. The model already helps. Now we move the line.

### 8.4 Cut-off 0.3

In [46]:
# Call a customer a leaver if the score is 0.3 or more
y_pred_03 = (scores >= 0.3).astype(int)

# The four boxes
tn_03, fp_03, fn_03, tp_03 = confusion_matrix(y_test, y_pred_03).ravel()

# The metrics
accuracy_03 = accuracy_score(y_test, y_pred_03)
precision_03 = tp_03 / (tp_03 + fp_03)
recall_03 = tp_03 / (tp_03 + fn_03)

# The bill
bill_03 = fp_03 * cost_false_alarm + fn_03 * cost_miss

print("Caught:", tp_03, "  False alarms:", fp_03, "  Missed:", fn_03, "  Correct no:", tn_03)
print("Accuracy:", round(accuracy_03, 3))
print("Precision:", round(precision_03, 3))
print("Recall:", round(recall_03, 3))
print("Bill: Rs", bill_03)

Caught: 364   False alarms: 320   Missed: 103   Correct no: 974
Accuracy: 0.76
Precision: 0.532
Recall: 0.779
Bill: Rs 778000


### 8.5 Cut-off 0.2

In [47]:
# Call a customer a leaver if the score is 0.2 or more
y_pred_02 = (scores >= 0.2).astype(int)

# The four boxes
tn_02, fp_02, fn_02, tp_02 = confusion_matrix(y_test, y_pred_02).ravel()

# The metrics
accuracy_02 = accuracy_score(y_test, y_pred_02)
precision_02 = tp_02 / (tp_02 + fp_02)
recall_02 = tp_02 / (tp_02 + fn_02)

# The bill
bill_02 = fp_02 * cost_false_alarm + fn_02 * cost_miss

print("Caught:", tp_02, "  False alarms:", fp_02, "  Missed:", fn_02, "  Correct no:", tn_02)
print("Accuracy:", round(accuracy_02, 3))
print("Precision:", round(precision_02, 3))
print("Recall:", round(recall_02, 3))
print("Bill: Rs", bill_02)

Caught: 408   False alarms: 474   Missed: 59   Correct no: 820
Accuracy: 0.697
Precision: 0.463
Recall: 0.874
Bill: Rs 591000


### 8.6 All Three Cut-offs Side by Side

In [48]:
summary = pd.DataFrame({
    "cut_off":      [0.5, 0.3, 0.2],
    "caught":       [tp_05, tp_03, tp_02],
    "false_alarms": [fp_05, fp_03, fp_02],
    "missed":       [fn_05, fn_03, fn_02],
    "accuracy":     [round(accuracy_05, 3), round(accuracy_03, 3), round(accuracy_02, 3)],
    "precision":    [round(precision_05, 3), round(precision_03, 3), round(precision_02, 3)],
    "recall":       [round(recall_05, 3), round(recall_03, 3), round(recall_02, 3)],
    "bill_rs":      [bill_05, bill_03, bill_02],
})

summary

,cut_off,caught,false_alarms,missed,accuracy,precision,recall,bill_rs
0,0.5,250,139,217,0.798,0.643,0.535,1371500
1,0.3,364,320,103,0.760,0.532,0.779,778000
2,0.2,408,474,59,0.697,0.463,0.874,591000


**Result.** As the line moves from 0.5 to 0.2, accuracy falls from about 79.8 to 69.7 percent. The bill falls from Rs 13.7 lakh to Rs 5.9 lakh. The most accurate line is the most expensive one.

### 8.7 The Decision

We choose **cut-off 0.2**. One miss costs as much as twelve false alarms, so catching more leavers is worth the extra offers.

**Three lines to the business owner (example):**

1. We recommend sending the Rs 500 offer to every customer the model scores at 0.2 or above.
2. On our test customers this reaches about 87 of every 100 customers who would leave, though about half the offers go to customers who would have stayed.
3. The total cost is about Rs 5.9 lakh, against Rs 13.7 lakh with the default line and Rs 28 lakh if we do nothing.